# U-Net 水下偏色图像增强 - Colab 训练配置和流程

这个 Notebook 用于将由于算力限制或本地开发完成的 U-Net 项目迁移到 Google Colab 的 GPU 上运行。

In [ ]:
# 1. 查看分配到的 GPU 资源
# 建议在菜单栏选择 "代码执行程序(Runtime)" -> "更改运行时类型" -> 硬件加速器选择 T4 GPU 或更高级 GPU
!nvidia-smi

In [ ]:
# 2. 挂载 Google Drive
# 我们将使用 Google Drive 来存放代码、数据集以及长期保存训练好的模型权重。
from google.colab import drive
drive.mount('/content/drive')

### 3. 项目和数据的准备

**方案 A: (推荐)** 将你本地的整个 `u-net` 文件夹压缩为 `unet_project.zip`，然后上传到 Google Drive 的根目录。用下面的代码解压：
```bash
!unzip -q -o /content/drive/MyDrive/unet_project.zip -d /content/drive/MyDrive/u-net
```

**方案 B:** 如果你已经把文件夹直接上传到了 Google Drive 里，那么直接让 Colab 进入这个目录即可。

In [ ]:
# 执行解压（请先将压缩包上传到 Google Drive 根目录名为 unet_project.zip）
!unzip -q -o /content/drive/MyDrive/unet_project.zip -d /content/drive/MyDrive/u-net

# 切换到该路径下 (这里假设你的项目就在 Google Drive 的 u-net 文件夹下，根据你的实际情况修改)
%cd /content/drive/MyDrive/u-net

# 查看目录结构，确保 dataset.py, train.py, model.py 等都在这
!ls -al

In [ ]:
# 4. 安装本地缺少的依赖项
# Colab 默认已经自带大部分常用库 (PyTorch, torchvision, PIL)
!pip install -r requirements.txt

In [ ]:
# 5. 执行模型训练代码
# 请确保当前目录下面有 data 文件夹，并且里面放好了 train/ 和 val/ 的输入和目标图。
# num_workers 在 Colab 上建议使用 2。如果出现内存不足等问题，可以把 num_workers 改为 0。
!python train.py \
    --mode train \
    --train_input_dir data/train/input \
    --train_target_dir data/train/target \
    --val_input_dir data/val/input \
    --val_target_dir data/val/target \
    --checkpoint_dir checkpoints \
    --max_epochs 150 \
    --batch_size 8 \
    --num_workers 2

In [ ]:
# 6. 推理与验证 (训练完成后)
# 指定一张测试图像路径
!python train.py --mode infer \
    --input_img data/val/input/test.jpg \
    --output_img output_enhanced.png \
    --ckpt checkpoints/best_model.pth

In [ ]:
# 7. 可视化查看推理前后的对比结果
import matplotlib.pyplot as plt
from PIL import Image

try:
    img_in = Image.open('data/val/input/test.jpg') # 替换为你真实测试的图片名字
    img_out = Image.open('output_enhanced.png')
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_in)
    axes[0].set_title("Input Image (Degraded)")
    axes[0].axis("off")

    axes[1].imshow(img_out)
    axes[1].set_title("Enhanced Output")
    axes[1].axis("off")

    plt.show()
except FileNotFoundError:
    print("请确保上面指定的测试图片路径存在哦")